# Multi-model ELI Skill & January Time Series

This notebook is the ELI counterpart of `4_refactor_sst_skill_ts.ipynb`. It compares CESM-SMYLE, E3SM-FOSIRL, and E3SM-Reanalysis using common target-year cohorts, then produces the matching ACC/nRMSE skill grid and 2-by-2 January-target time-series figure. ELI remains separate from rectangular SST indices because it is a longitude centroid with different units and interpretation.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
import re
import uuid
from pathlib import Path

# Resolve native-library paths from the active kernel.
_env_prefix = sys.prefix
_proj_path  = os.path.join(_env_prefix, "share", "proj")
if os.path.isfile(os.path.join(_proj_path, "proj.db")):
    os.environ["CONDA_PREFIX"] = _env_prefix
    os.environ["PROJ_LIB"]    = _proj_path
    os.environ["PROJ_DATA"]   = _proj_path

import dask
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy.stats import pearsonr

from esp_lab import stats
from esp_lab import eli_diagnostics as eli_tools
from esp_lab.utils import mov_utils as mov
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

print(f"Python      : {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set)')}")


Python      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/homes/z/zhan391/.conda/envs/e3sm_analysis


## Dask cluster

Create and own one cluster for lazy NetCDF reads, drift removal, and skill computation. Re-running this cell first closes resources left by the previous run, following the lifecycle used by `1a_refactor_atm_leadtime_acc_skill_map.ipynb`.


In [2]:
# Shared login/Jupyter nodes are capped at four local workers by DaskConfig.
machine_env = os.environ.get("CLUSTER_TYPE", "local")
dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=4,
    cores=1,
    memory_limit="4GB",
    walltime="02:00:00",
)
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(), lambda: get_cluster_client(dask_cfg)
)
print(client)
print("xarray:", xr.__version__)
print("dask:", dask.__version__)



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
This can easily exceed memory/CPU limits and crash your Jupyter kernel.
We highly recommend launching Jupyter on an Exclusive Perlmutter Compute Node.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Scheduler: tcp://127.0.0.1:43335
Connected workers: 4
Dashboard: http://127.0.0.1:8787/status
<Client: 'tcp://127.0.0.1:43335' processes=4 threads=4, memory=14.90 GiB>
xarray: 2026.4.0
dask: 2026.3.0


## Configuration

Edit the compact `WORKFLOW_SETTINGS` block below to select years, initialization months, ELI grid, paths, and models. Derived paths and model specifications are built automatically, following the centralized configuration pattern used by the 1a workflow. All configured model/month caches remain required so a comparison cannot silently omit a hindcast.


In [3]:
# User-facing workflow configuration. Later cells consume derived aliases.
INDEX = "ELI"
YEAR_END = 2011  # Set to None to use initialization_years[1].

WORKFLOW_SETTINGS = {
    "paths": {
        "diag_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
        "figure_outdir": "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag",
        "obs_eli_file": "/global/cfs/cdirs/e3sm/lvroekel/ELI_ERSSTv5_1854.01-2020.05.xlsx",
    },
    "run": {
        "initialization_years": (1980, 2018),
        "year_end": YEAR_END,
        "init_months": [5, 11],
        "climatology_years": (1981, 2010),
    },
    "inputs": {
        "eli_grid": "native",  # native | regridded
        "monthly_nlead": 24,
        "nmme_period": "1980_2020",
        # Keep leads independent while reductions span years and members.
        "chunks": {"Y": -1, "L": 1, "M": -1},
    },
    "models": {
        "CESM-SMYLE": {"label": "CESM-SMYLE", "cache_tag": "CESM-SMYLE", "ensemble_size": 20, "color": "tab:orange", "marker": "s", "family": "smyle"},
        "E3SM-FOSIRL": {"label": "E3SMv3-FOSIRL", "cache_tag": "JRA55_FOSIRL", "ensemble_size": 10, "color": "black", "marker": "o", "family": "e3sm"},
        "E3SM-Reanalysis": {"label": "E3SMv3-Reanalysis", "cache_tag": "Reanalysis", "ensemble_size": 10, "color": "tab:blue", "marker": "^", "family": "e3sm"},
        "E3SM-4DEnVarOcn": {"label": "E3SMv3-4DEnVarOcn", "cache_tag": "4DEnVarOcn", "ensemble_size": 10, "color": "tab:purple", "marker": "D", "family": "e3sm"},
    },
}

PATHS = WORKFLOW_SETTINGS["paths"]
RUN = WORKFLOW_SETTINGS["run"]
INPUTS = WORKFLOW_SETTINGS["inputs"]
DIAG_ROOT = Path(PATHS["diag_root"])
FIGURE_OUTDIR = Path(PATHS["figure_outdir"])
OBS_ELI_FILE = Path(PATHS["obs_eli_file"])
START_YEAR, configured_end_year = map(int, RUN["initialization_years"])
END_YEAR = configured_end_year if RUN["year_end"] is None else int(RUN["year_end"])
INIT_MONTHS = list(map(int, RUN["init_months"]))
CLIM_Y0, CLIM_Y1 = map(int, RUN["climatology_years"])
NLEAD = int(INPUTS["monthly_nlead"])
ELI_INPUT_GRID = str(INPUTS["eli_grid"])
NMME_PERIOD_TAG = str(INPUTS["nmme_period"])
ELI_CHUNKS = dict(INPUTS["chunks"])
if ELI_INPUT_GRID not in {"native", "regridded"}:
    raise ValueError("inputs.eli_grid must be 'native' or 'regridded'")
if not START_YEAR <= END_YEAR <= configured_end_year:
    raise ValueError("run.year_end must lie within run.initialization_years")

NMME_OUTDIR = DIAG_ROOT / "NMME"
NMME_ELI_SKILL_FILE = NMME_OUTDIR / "sst_index" / "skill" / f"NMME_{INDEX}_skill_{NMME_PERIOD_TAG}.nc"
MULTIMODEL_ELI_DIR = DIAG_ROOT / "multimodel" / "eli"
for directory in (FIGURE_OUTDIR, MULTIMODEL_ELI_DIR):
    directory.mkdir(parents=True, exist_ok=True)
NMME_ELI_TIMESERIES_FILES = {
    month: NMME_OUTDIR / "sst_index" / "timeseries" / f"NMME{month:02d}_{INDEX}_mon_dd_{NMME_PERIOD_TAG}.nc"
    for month in INIT_MONTHS
}


e3sm_template = (
    "E3SMLE{init_month:02d}_{INDEX}_native_N{nens:02d}_M{nlead:02d}.nc"
    if ELI_INPUT_GRID == "native"
    else "E3SMLE{init_month:02d}_{INDEX}_N{nens:02d}_M{nlead:02d}.nc"
)
MODEL_SPECS = {
    key: eli_tools.ELIModelSpec(
        key=key,
        label=entry["label"],
        root=DIAG_ROOT / entry["cache_tag"] / "sst_index" / "timeseries",
        filename_template=(
            "BSMYLE{init_month:02d}_{INDEX}_N{nens:02d}_M{nlead:02d}.nc"
            if entry["family"] == "smyle" else e3sm_template
        ),
        ensemble_size=int(entry["ensemble_size"]),
        color=entry["color"],
        marker=entry["marker"],
    )
    for key, entry in WORKFLOW_SETTINGS["models"].items()
}

# -----------------------------------------------------------------------------
# Figure configuration
# -----------------------------------------------------------------------------
MODEL_LINESTYLES = {model: ("--" if model == "CESM-SMYLE" else "-") for model in MODEL_SPECS}
NMME_STYLE = {"label": "NMME MMM", "color": "tab:red", "linestyle": "-", "marker": "^", "spread_alpha": 0.18}

SKILL_PLOT_CONFIG = {
    "filename_parts": (INDEX, "multimodel", "acc_nrmse_skill"),
    "figfmt": "png",
    "dpi": 300,
    "font_size": 16,
    "title_font_scale": 1.05,
    "header_font_scale": 1.15,
    "tick_font_scale": 0.95,
    "legend_font_scale": 0.90,
    "line_width": 2.8,
    "marker_size_scale": 0.60,
    "significance_pval": 0.1,
    "hindcast_labels": {2: "FEB init", 5: "MAY init", 8: "AUG init", 11: "NOV init"},
    "all_init_label": "ALL init average",
    "monthly_xticks": np.arange(1, NLEAD, 2),
    "monthly_xticks_minor": np.arange(0, NLEAD, 2),
    "monthly_xlim": [-0.5, NLEAD],
    "acc_ylim": [-0.2, 1.0],
    "nrmse_ylim": [0.0, 1.8],
    "acc_reference": 0.5,
    "nrmse_reference": 1.0,
    "reference_color": "black",
    "ncol": 2,
    "fig_width": 14,
    "row_height_scale": 0.60,
    "acc_header": "Anomaly Correlation Coefficient",
    "nrmse_header": "Normalized Root Mean Square Error",
    "header_y": 1.12,
    "reference_line_width": 1,
    "show_grid": True,
    "acc_ylabel": "ACC",
    "nrmse_ylabel": "nRMSE",
    "xlabel": "Lead month",
    "legend_loc": "lower center",
    "legend_bbox": (0.5, 0.01),
    "legend_facecolor": "#f9f9f9",
    "legend_edgecolor": "lightgray",
    "legend_framealpha": 0.9,
    "layout_rect": [0, 0.06, 1, 0.97],
    "metric": "eli_multimodel_skill",
    "title": f"Multi-model {INDEX} Skill",
}

TIMESERIES_PLOT_CONFIG = {
    "filename_parts": (INDEX, "multimodel", "time_series"),
    "figfmt": "png",
    "dpi": 300,
    "target_leads": {5: [8, 20], 11: [2, 14]},
    "ncol": 2,
    "font_size": 16,
    "title_font_scale": 1.05,
    "tick_font_scale": 0.90,
    "legend_font_scale": 0.90,
    "annotation_font_scale": 0.68,
    "fig_width": 14,
    "row_height_scale": 0.60,
    "line_width": 2.6,
    "obs_color": "0.25",
    "obs_marker": ".",
    "obs_marker_size": 7,
    "spread_alpha": 0.10,
    "annotation_start_y": 0.03,
    "annotation_step_y": 0.075,
    "annotation_x": 0.02,
    "annotation_box": {"facecolor": "white", "edgecolor": "0.8", "alpha": 0.85},
    "plot_ylim": [-20, 30],
    "major_yticks": np.arange(-20, 31, 10),
    "minor_yticks": np.arange(-20, 31, 5),
    "major_years": np.arange(1980, 2021, 10),
    "minor_years": np.arange(1980, 2021, 5),
    "hindcast_labels": {5: "MAY", 11: "NOV"},
    "figlabs": [["(a)", "(c)"], ["(b)", "(d)"]],
    "minor_grid_alpha": 0.25,
    "show_major_grid": True,
    "axis_year_margin": 1,
    "ylabel": "ELI anomaly (degE)",
    "xlabel": "Target year",
    "legend_loc": "lower center",
    "legend_bbox": (0.5, 0.01),
    "legend_edgecolor": "lightgray",
    "legend_framealpha": 0.9,
    "layout_rect": [0, 0.07, 1, 1],
    "metric": "eli_multimodel_time_series",
    "title": f"Multi-model {INDEX} Time Series",
}

print(f"Evaluation years: {START_YEAR}-{END_YEAR}")
print(f"Climatology: {CLIM_Y0}-{CLIM_Y1}")
print(f"Initialization months: {INIT_MONTHS}")
print(f"ELI input grid: {ELI_INPUT_GRID}")
print("Models:", [spec.label for spec in MODEL_SPECS.values()])

Evaluation years: 1980-2011
Climatology: 1981-2010
Initialization months: [5, 11]
ELI input grid: native
Models: ['CESM-SMYLE', 'E3SMv3-FOSIRL', 'E3SMv3-Reanalysis', 'E3SMv3-4DEnVarOcn']


## Validate required inputs


In [4]:
if not OBS_ELI_FILE.is_file():
    raise FileNotFoundError(f"Observational ELI file not found: {OBS_ELI_FILE}")

required_model_files = {
    model: [spec.path(month, NLEAD) for month in INIT_MONTHS]
    for model, spec in MODEL_SPECS.items()
}
missing_model_files = [
    path
    for paths in required_model_files.values()
    for path in paths
    if not path.is_file()
]
if missing_model_files:
    raise FileNotFoundError(
        "Missing required ELI caches:\n" +
        "\n".join(f"  {path}" for path in missing_model_files)
    )

NMME_ELI_SKILL_AVAILABLE = NMME_ELI_SKILL_FILE.is_file()
nmme_skill_plot = None
nmme_skill_spread = None
if NMME_ELI_SKILL_AVAILABLE:
    nmme_eli_skill = workflow_resources.track(
        xr.open_dataset(NMME_ELI_SKILL_FILE, chunks="auto")
    )
    nmme_skill_plot = xr.Dataset({
        "acc": nmme_eli_skill["nmme_skill_mmm_corr"],
        "pval": nmme_eli_skill["nmme_skill_mmm_pval"],
        # The archived NMME `rmse` follows the Yeager normalized-RMSE convention.
        "nrmse": nmme_eli_skill["nmme_skill_mmm_rmse"],
    }).rename({"monthly_L": "L"})
    nmme_skill_spread = xr.Dataset({
        "acc": nmme_eli_skill["nmme_skill_corr"],
        "nrmse": nmme_eli_skill["nmme_skill_rmse"],
    }).rename({"monthly_L": "L"})
NMME_ELI_TIMESERIES_AVAILABLE = all(path.is_file() for path in NMME_ELI_TIMESERIES_FILES.values())
nmme_eli_timeseries = {}
if NMME_ELI_TIMESERIES_AVAILABLE:
    for month, path in NMME_ELI_TIMESERIES_FILES.items():
        ds = workflow_resources.track(xr.open_dataset(path, chunks="auto"))
        ds = ds.sel(Y=slice(START_YEAR, END_YEAR))
        nmme_eli_timeseries[month] = {"values": ds["sst"], "time": ds["time"]}
print("Configuration valid")
for model, paths in required_model_files.items():
    print(f"  {MODEL_SPECS[model].label}:")
    for path in paths:
        print(f"    {path.name}")
print(f"  NMME skill available: {NMME_ELI_SKILL_AVAILABLE}")
print(f"  NMME time series available: {NMME_ELI_TIMESERIES_AVAILABLE}")


KeyError: 'INDEX'

## 1.  Load observational ELI

The Excel file contains a monthly ELI time series derived from ERSSTv5.
Each column header is a year; each row is a calendar month (0-indexed).
Values are converted to an `xarray.DataArray` with a monthly time axis.

In [ ]:
df_obs = pd.read_excel(OBS_ELI_FILE)

# Collect all available years in column order.
obs_years = [c for c in df_obs.columns if isinstance(c, (int, np.integer))]

times_obs    = []
eli_obs_vals = []
for yr in obs_years:
    for mo in range(12):
        val = df_obs[yr].iloc[mo]
        times_obs.append(pd.Timestamp(year=int(yr), month=mo + 1, day=1))
        eli_obs_vals.append(float(val) if pd.notna(val) else np.nan)

obs_time = pd.DatetimeIndex(times_obs)
eli_obs = xr.DataArray(
    np.array(eli_obs_vals, dtype=np.float32),
    dims="time",
    coords={"time": obs_time},
    name="eli_obs",
    attrs={
        "long_name": "Observed Equatorial Longitude Index (ERSSTv5)",
        "units": "degrees_east",
        "source": str(OBS_ELI_FILE),
    },
)

# Clip to model verification period (with buffer for drift removal).
eli_obs = eli_obs.sel(time=slice(str(START_YEAR - 2), str(END_YEAR + 2)))

# Monthly climatology over the base period for anomaly computation.
obs_clim_mask = (
    (eli_obs.time.dt.year >= CLIM_Y0) &
    (eli_obs.time.dt.year <= CLIM_Y1)
)
obs_clim = eli_obs.isel(time=obs_clim_mask).groupby("time.month").mean()

eli_obs_anom = eli_obs.groupby("time.month") - obs_clim
eli_obs_anom.name = "eli_obs_anom"
eli_obs_anom.attrs = {
    "long_name": "Observed ELI anomaly (ERSSTv5)",
    "units": "degrees_east",
}

print(f"Observational ELI: {len(eli_obs)} months")
print(f"  period : {str(eli_obs.time.values[0])[:10]} — {str(eli_obs.time.values[-1])[:10]}")
print(f"  min/max: {float(eli_obs.min()):.2f} / {float(eli_obs.max()):.2f}°E")
print(f"  mean   : {float(eli_obs.mean()):.2f}°E")
print(f"\nObs anomaly std : {float(eli_obs_anom.std()):.2f}°E")

## 2. Load CESM-SMYLE and both E3SM hindcasts


In [ ]:
eli_model, eli_time, eli_provenance = eli_tools.load_model_hindcasts(
    MODEL_SPECS,
    INIT_MONTHS,
    nlead=NLEAD,
    start_year=START_YEAR,
    end_year=END_YEAR,
    chunks=ELI_CHUNKS,
    resource_tracker=workflow_resources,
)

for model, spec in MODEL_SPECS.items():
    for month in INIT_MONTHS:
        values = eli_model[model][month]
        print(
            f"{spec.label}, init {month:02d}: sizes={dict(values.sizes)}, "
            f"finite={float(values.notnull().mean()):.3f}, "
            f"range={float(values.min()):.2f}-{float(values.max()):.2f} degE"
        )


## 3. Lead-dependent drift removal

Each model uses its own 1981–2010 ensemble climatology at every lead. This removes model drift while preserving a common verification cohort for the later comparison.


In [ ]:
eli_dd, eli_drift = eli_tools.remove_lead_drift(
    eli_model,
    clim_start=CLIM_Y0,
    clim_end=CLIM_Y1,
)

# Drift anomalies are reused for cohort selection, skill, and plotting.
# Persist once so those consumers share the distributed result.
if client is not None:
    persisted_keys = [
        (model, month) for model in MODEL_SPECS for month in INIT_MONTHS
    ]
    persisted_values = client.persist([
        eli_dd[model][month] for model, month in persisted_keys
    ])
    for (model, month), values in zip(persisted_keys, persisted_values):
        eli_dd[model][month] = values
    from dask.distributed import wait
    wait(persisted_values)

for model, spec in MODEL_SPECS.items():
    for month in INIT_MONTHS:
        drift = eli_drift[model][month]
        print(
            f"{spec.label}, init {month:02d}: drift "
            f"{float(drift.min()):.2f}-{float(drift.max()):.2f} degE"
        )


## 4. Common-cohort skill computation

ACC, p values, RMSE, and normalized RMSE are calculated on the intersection of valid target years across CESM-SMYLE, both E3SM hindcasts, and ERSSTv5 independently for every initialization month and lead.


In [ ]:
skill, common_target_years = eli_tools.compute_multimodel_skill(
    eli_dd,
    eli_time,
    eli_obs_anom,
    INIT_MONTHS,
)

for month in INIT_MONTHS:
    print(
        f"Common samples, init {month:02d}:",
        {
            lead: (years[0], years[-1], len(years))
            for lead, years in common_target_years[month].items()
        },
    )

model_coord = xr.DataArray(list(MODEL_SPECS), dims="model", name="model")
startmonth_coord = xr.DataArray(INIT_MONTHS, dims="startmonth", name="startmonth")
skill_dataset = xr.concat(
    [
        xr.concat(
            [skill[model][month] for month in INIT_MONTHS],
            dim=startmonth_coord,
            join="exact",
        )
        for model in MODEL_SPECS
    ],
    dim=model_coord,
    join="exact",
)
skill_dataset.attrs = {}
skill_dataset.attrs.update(
    description="CESM-SMYLE and two E3SM hindcast ELI skill on common target-year cohorts",
    verification_initialization_years=f"{START_YEAR}-{END_YEAR}",
    climatology=f"{CLIM_Y0}-{CLIM_Y1}",
    sample_alignment="intersection across all compared models and ERSSTv5 independently by initialization and lead",
    models=",".join(MODEL_SPECS),
    eli_input_grid=ELI_INPUT_GRID,
)
skill_file = MULTIMODEL_ELI_DIR / f"eli_multimodel_skill_{START_YEAR}_{END_YEAR}.nc"
temporary = skill_file.with_name(f".{skill_file.name}.tmp.{uuid.uuid4().hex}")
try:
    skill_dataset.to_netcdf(temporary)
    os.replace(temporary, skill_file)
finally:
    temporary.unlink(missing_ok=True)
print("Saved:", skill_file)
display(skill_dataset)


## 5. Multi-model ACC and normalized-RMSE skill

This follows the skill-figure layout in `4_refactor_sst_skill_ts.ipynb`: one row per initialization month, followed by the monthly all-initialization mean. Filled ACC markers indicate `p < 0.1`; open markers show the remaining leads. CESM-SMYLE and both E3SM systems use the same target-year cohort at each initialization and lead. The red NMME MMM benchmark and its across-model range come from the archived 1980–2020 NMME ELI skill product and cover leads 1–12.


In [ ]:

def figure_filename(*parts, ext="png"):
    clean = ["fig"]
    for part in parts:
        token = re.sub(r"[^A-Za-z0-9]+", "_", str(part).strip()).strip("_").lower()
        if token:
            clean.append(token)
    return "_".join(clean) + f".{ext.lstrip('.')}"

# -----------------------------
# Plot monthly ELI skill scores using the layout from 4_refactor_sst_skill_ts.ipynb
# -----------------------------
plot_cfg = SKILL_PLOT_CONFIG
nmme_style = NMME_STYLE
figname = figure_filename(*plot_cfg["filename_parts"], ext=plot_cfg["figfmt"])
fontz = plot_cfg["font_size"]
title_fontz = fontz * plot_cfg["title_font_scale"]
header_fontz = fontz * plot_cfg["header_font_scale"]
tick_fontz = fontz * plot_cfg["tick_font_scale"]
legend_fontz = fontz * plot_cfg["legend_font_scale"]
line_width = plot_cfg["line_width"]
marker_size = fontz * plot_cfg["marker_size_scale"]

plt.rcParams.update({
    "font.size": fontz,
    "axes.titlesize": title_fontz,
    "xtick.labelsize": tick_fontz,
    "ytick.labelsize": tick_fontz,
    "legend.fontsize": legend_fontz,
})

nrow, ncol = len(INIT_MONTHS) + 1, plot_cfg["ncol"]
fig_width = plot_cfg["fig_width"]
row_height = fig_width * plot_cfg["row_height_scale"] / ncol
fig = plt.figure(figsize=(fig_width, row_height * nrow))
figlabs = [f"({chr(97 + i)})" for i in range(nrow * ncol)]

skill_by_model = {
    model: xr.concat(
        [skill[model][month] for month in INIT_MONTHS],
        dim=xr.DataArray(INIT_MONTHS, dims="startmonth", name="startmonth"),
        join="exact",
    )
    for model in MODEL_SPECS
}

for row, month in enumerate(INIT_MONTHS + [None]):
    ax = fig.add_subplot(nrow, ncol, row * 2 + 1)
    ax2 = fig.add_subplot(nrow, ncol, row * 2 + 2)
    row_label = plot_cfg["all_init_label"] if month is None else plot_cfg["hindcast_labels"].get(month, f"{month:02d} init")
    ax.set_title(f"{figlabs[row * 2]} {row_label}", loc="left", fontsize=title_fontz)
    ax2.set_title(f"{figlabs[row * 2 + 1]} {row_label}", loc="left", fontsize=title_fontz)
    if row == 0:
        ax.set_title(plot_cfg["acc_header"], loc="center", fontsize=header_fontz, y=plot_cfg["header_y"])
        ax2.set_title(plot_cfg["nrmse_header"], loc="center", fontsize=header_fontz, y=plot_cfg["header_y"])

    for model, spec in MODEL_SPECS.items():
        scores = skill_by_model[model].mean("startmonth") if month is None else skill[model][month]
        x = scores.L - 1
        linestyle = MODEL_LINESTYLES[model]

        ax.plot(
            x, scores.acc,
            color=spec.color,
            linewidth=line_width,
            linestyle=linestyle,
            label=spec.label,
        )
        ax.plot(
            x, scores.acc,
            color=spec.color,
            linestyle="none",
            marker=spec.marker,
            markerfacecolor="none",
            markersize=marker_size,
        )
        # A mean p value is not a valid all-init significance test, so only
        # initialization-specific rows receive filled significance markers.
        if month is not None:
            ax.plot(
                x, scores.acc.where(scores.pval < plot_cfg["significance_pval"]),
                color=spec.color,
                linestyle="none",
                marker=spec.marker,
                markersize=marker_size,
            )

        ax2.plot(
            x, scores.nrmse,
            color=spec.color,
            linewidth=line_width,
            linestyle=linestyle,
            marker=spec.marker,
            markersize=marker_size,
            label=spec.label,
        )

    if nmme_skill_plot is not None:
        if month is None:
            nmme_scores = nmme_skill_plot.mean("startmonth")
            nmme_range = nmme_skill_spread
            spread_dims = [dim for dim in ("startmonth", "model") if dim in nmme_range.dims]
        elif month in nmme_skill_plot.startmonth.values:
            nmme_scores = nmme_skill_plot.sel(startmonth=month)
            nmme_range = nmme_skill_spread.sel(startmonth=month)
            spread_dims = [dim for dim in ("model",) if dim in nmme_range.dims]
        else:
            nmme_scores = None

        if nmme_scores is not None:
            x_nmme = nmme_scores.L - 1
            ax.fill_between(
                x_nmme,
                nmme_range.acc.min(spread_dims, skipna=True),
                nmme_range.acc.max(spread_dims, skipna=True),
                color=nmme_style["color"], alpha=nmme_style["spread_alpha"], linewidth=0,
            )
            ax2.fill_between(
                x_nmme,
                nmme_range.nrmse.min(spread_dims, skipna=True),
                nmme_range.nrmse.max(spread_dims, skipna=True),
                color=nmme_style["color"], alpha=nmme_style["spread_alpha"], linewidth=0,
            )
            ax.plot(
                x_nmme, nmme_scores.acc,
                color=nmme_style["color"], linewidth=line_width, linestyle=nmme_style["linestyle"],
                marker=nmme_style["marker"], markersize=marker_size, label=nmme_style["label"],
            )
            if month is not None:
                ax.plot(
                    x_nmme, nmme_scores.acc.where(nmme_scores.pval < plot_cfg["significance_pval"]),
                    color=nmme_style["color"], linestyle="none", marker=nmme_style["marker"],
                    markersize=marker_size,
                )
            ax2.plot(
                x_nmme, nmme_scores.nrmse,
                color=nmme_style["color"], linewidth=line_width, linestyle=nmme_style["linestyle"],
                marker=nmme_style["marker"], markersize=marker_size, label=nmme_style["label"],
            )

    for panel, ylim, reference in ((ax, plot_cfg["acc_ylim"], plot_cfg["acc_reference"]), (ax2, plot_cfg["nrmse_ylim"], plot_cfg["nrmse_reference"])):
        panel.set_xticks(plot_cfg["monthly_xticks"])
        panel.set_xticks(plot_cfg["monthly_xticks_minor"], minor=True)
        panel.set_xlim(plot_cfg["monthly_xlim"])
        panel.set_ylim(ylim)
        panel.tick_params(axis="both", labelsize=tick_fontz)
        panel.grid(plot_cfg["show_grid"])
        panel.axhline(reference, color=plot_cfg["reference_color"], linewidth=plot_cfg["reference_line_width"])

    ax.set_ylabel(plot_cfg["acc_ylabel"], fontsize=tick_fontz)
    ax2.set_ylabel(plot_cfg["nrmse_ylabel"], fontsize=tick_fontz)
    if row == nrow - 1:
        ax.set_xlabel(plot_cfg["xlabel"], fontsize=tick_fontz)
        ax2.set_xlabel(plot_cfg["xlabel"], fontsize=tick_fontz)

handles, labels = fig.axes[0].get_legend_handles_labels()
if nmme_skill_spread is not None and nmme_style["label"] in labels:
    handles.append(mpatches.Patch(facecolor=nmme_style["color"], alpha=nmme_style["spread_alpha"], linewidth=0))
    labels.append("NMME range")
fig.legend(
    handles, labels,
    loc=plot_cfg["legend_loc"],
    ncol=len(labels),
    bbox_to_anchor=plot_cfg["legend_bbox"],
    frameon=True,
    facecolor=plot_cfg["legend_facecolor"],
    edgecolor=plot_cfg["legend_edgecolor"],
    framealpha=plot_cfg["legend_framealpha"],
)
fig.tight_layout(rect=plot_cfg["layout_rect"])
figpath = FIGURE_OUTDIR / figname
mov.save_figure(
    fig, figpath,
    mode="",
    metric=plot_cfg["metric"],
    title=plot_cfg["title"],
    caption=(
        f"Common target-year cohorts for CESM-SMYLE and E3SM; "
        f"NMME benchmark period {NMME_PERIOD_TAG}"
    ),
    dpi=plot_cfg["dpi"],
)
print("Saved figure:", figpath)
plt.show()


## 6. January-target ELI time series

This follows the 2-by-2 target-season layout in `4_refactor_sst_skill_ts.ipynb`. Each panel shows ERSSTv5 and the three forecast-system ensemble means and spreads. NMME MMM and its across-model spread are overlaid where the 12-lead NMME archive reaches the requested January target. Primary-system skill annotations use the same common target-year cohort as the lead-time skill figure; NMME annotations use the archived 1980–2020 skill product.


In [ ]:
# -----------------------------
# Plot January-target ELI time series using the layout from 4_refactor_sst_skill_ts.ipynb
# -----------------------------
plot_cfg = TIMESERIES_PLOT_CONFIG
nmme_style = NMME_STYLE
target_leads = plot_cfg["target_leads"]
plot_months = [month for month in INIT_MONTHS if month in target_leads]
nrow, ncol = len(plot_months), plot_cfg["ncol"]

fontz = plot_cfg["font_size"]
title_fontz = fontz * plot_cfg["title_font_scale"]
tick_fontz = fontz * plot_cfg["tick_font_scale"]
legend_fontz = fontz * plot_cfg["legend_font_scale"]
annotation_fontz = fontz * plot_cfg["annotation_font_scale"]
fig_width = plot_cfg["fig_width"]
row_height = fig_width * plot_cfg["row_height_scale"] / ncol
line_width = plot_cfg["line_width"]

plt.rcParams.update({
    "font.size": fontz,
    "axes.titlesize": title_fontz,
    "xtick.labelsize": tick_fontz,
    "ytick.labelsize": tick_fontz,
    "legend.fontsize": legend_fontz,
})

fig = plt.figure(figsize=(fig_width, row_height * nrow))
for row, month in enumerate(plot_months):
    for col, lead in enumerate(target_leads[month]):
        ax = fig.add_subplot(nrow, ncol, row * 2 + col + 1)
        reference_times = eli_time[next(iter(MODEL_SPECS))][month].sel(L=lead)
        target_years = np.asarray(reference_times.dt.year.values, dtype=float)
        target_times = np.asarray(reference_times.values, dtype=object)
        obs_values = eli_tools.observations_for_times(eli_obs_anom, target_times)
        valid_obs = np.isfinite(target_years) & np.isfinite(obs_values)

        ax.plot(
            target_years[valid_obs], obs_values[valid_obs],
            color=plot_cfg["obs_color"],
            linewidth=line_width,
            marker=plot_cfg["obs_marker"],
            markersize=plot_cfg["obs_marker_size"],
            label="ERSSTv5",
        )

        annotation_y = plot_cfg["annotation_start_y"]
        for model, spec in MODEL_SPECS.items():
            values = eli_dd[model][month].sel(L=lead)
            mean = values.mean("M")
            spread = values.std("M")
            valid_model = np.isfinite(target_years) & np.isfinite(mean.values)
            years_model = target_years[valid_model]
            mean_model = mean.values[valid_model]
            spread_model = spread.values[valid_model]
            linestyle = MODEL_LINESTYLES[model]

            ax.plot(
                years_model, mean_model,
                color=spec.color,
                linewidth=line_width,
                linestyle=linestyle,
                label=spec.label,
            )
            ax.fill_between(
                years_model,
                mean_model - spread_model,
                mean_model + spread_model,
                color=spec.color,
                alpha=plot_cfg["spread_alpha"],
                linewidth=0,
            )

            score = skill[model][month].sel(L=lead)
            ax.text(
                plot_cfg["annotation_x"], annotation_y,
                f"{spec.label} ACC={float(score.acc):.2f}, nRMSE={float(score.nrmse):.2f}",
                transform=ax.transAxes,
                fontsize=annotation_fontz,
                color=spec.color,
                bbox=plot_cfg["annotation_box"],
            )
            annotation_y += plot_cfg["annotation_step_y"]

        if month in nmme_eli_timeseries and lead in nmme_eli_timeseries[month]["values"].L.values:
            nmme_values = nmme_eli_timeseries[month]["values"].sel(L=lead)
            nmme_times = nmme_eli_timeseries[month]["time"].sel(L=lead)
            nmme_years = np.asarray(nmme_times.dt.year.values, dtype=float)
            nmme_model_means = nmme_values.mean("M", skipna=True)
            nmme_mean = nmme_model_means.mean("model", skipna=True)
            nmme_spread = nmme_model_means.std("model", skipna=True)
            nmme_valid = np.isfinite(nmme_years) & np.isfinite(nmme_mean.values)
            ax.plot(
                nmme_years[nmme_valid], nmme_mean.values[nmme_valid],
                color=nmme_style["color"], linewidth=line_width, linestyle=nmme_style["linestyle"],
                label=nmme_style["label"],
            )
            ax.fill_between(
                nmme_years[nmme_valid],
                (nmme_mean - nmme_spread).values[nmme_valid],
                (nmme_mean + nmme_spread).values[nmme_valid],
                color=nmme_style["color"], alpha=nmme_style["spread_alpha"], linewidth=0,
            )

        lead_months = int(lead) - 2
        figlabel = plot_cfg["figlabs"][row][col] if len(plot_months) == 2 else f"({chr(97 + row * 2 + col)})"
        ax.set_title(
            f"{figlabel} {plot_cfg['hindcast_labels'].get(month, f'{month:02d}')} init ({lead_months}-mon lead)",
            loc="left",
            fontsize=title_fontz,
        )
        ax.set_xlim([plot_cfg["major_years"].min() - plot_cfg["axis_year_margin"], plot_cfg["major_years"].max() + plot_cfg["axis_year_margin"]])
        ax.set_xticks(plot_cfg["major_years"])
        ax.set_xticks(plot_cfg["minor_years"], minor=True)
        ax.set_ylim(plot_cfg["plot_ylim"])
        ax.set_yticks(plot_cfg["major_yticks"])
        ax.set_yticks(plot_cfg["minor_yticks"], minor=True)
        ax.tick_params(axis="both", labelsize=tick_fontz)
        ax.grid(plot_cfg["show_major_grid"], which="major")
        ax.grid(True, which="minor", alpha=plot_cfg["minor_grid_alpha"])
        if col == 0:
            ax.set_ylabel(plot_cfg["ylabel"], fontsize=tick_fontz)
        if row == nrow - 1:
            ax.set_xlabel(plot_cfg["xlabel"], fontsize=tick_fontz)

handles, labels = fig.axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc=plot_cfg["legend_loc"],
    ncol=len(labels),
    bbox_to_anchor=plot_cfg["legend_bbox"],
    edgecolor=plot_cfg["legend_edgecolor"],
    framealpha=plot_cfg["legend_framealpha"],
)
fig.tight_layout(rect=plot_cfg["layout_rect"])
figpath = FIGURE_OUTDIR / figure_filename(*plot_cfg["filename_parts"], ext=plot_cfg["figfmt"])
mov.save_figure(
    fig, figpath,
    mode="",
    metric=plot_cfg["metric"],
    title=plot_cfg["title"],
    caption=(
        f"Displayed initialization years: {START_YEAR}-{END_YEAR}; "
        f"primary-system annotations use common cohorts; NMME skill period {NMME_PERIOD_TAG}"
    ),
    dpi=plot_cfg["dpi"],
)
print("Saved figure:", figpath)
plt.show()


## Clean up


In [ ]:
# Release lazy NetCDF handles and this notebook's Dask resources.
close_notebook_resources(globals())
print("Closed ELI datasets and Dask resources.")
